In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime

SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""


@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city"""
    return f"It's always sunny in {city}"


@dataclass
class Context:
    """Custom runtime context schema."""

    user_id: str


@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return 'Florida' if user_id == '1' else 'SF'


model = init_chat_model(
    'deepseek-chat',
    temperature=0.5,
    timeout=10,
    max_tokens=1024,
)

@dataclass
class ResponseFormat:
    """Response schema for the agent."""

    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None


checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer,
)

In [3]:
config = {'configurable': {'thread_id': '1'}}
response1 = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'what is the weather outside?'}]},
    config=config,  # type: ignore
    context=Context(user_id='1'),
)

In [7]:
from pprint import pprint

pprint(response1)

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='b4ef3440-de5a-44bd-9c35-1b454f059003'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 519, 'total_tokens': 540, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 519}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '6f5312c9-6050-4ea6-9d84-dfe4bdd6395c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d42af-50c5-77c0-af85-cf4a26b37ac6-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_00_wIvbkKmLSZOT7DCvyP6ju0RM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 519, 'output_tokens': 21, 'total_tokens': 540

In [9]:
response2 = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'thank you!'}]},
    config=config,  # type: ignore
    context=Context(user_id='1'),
)

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


In [10]:
pprint(response2)

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='b4ef3440-de5a-44bd-9c35-1b454f059003'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 519, 'total_tokens': 540, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 519}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '6f5312c9-6050-4ea6-9d84-dfe4bdd6395c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d42af-50c5-77c0-af85-cf4a26b37ac6-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_00_wIvbkKmLSZOT7DCvyP6ju0RM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 519, 'output_tokens': 21, 'total_tokens': 540